# Tennis Votes Preprocess

## 1. Setup and locate the Votes data

### 1.1 Define paths and collect Votes files

In [1]:
from pathlib import Path
import polars as pl

MINI_PROJECT_ROOT = Path.cwd().parents[2]

DATA_ROOT = (
    MINI_PROJECT_ROOT
    / "Tennis Schema"
    / "tennis_data"
)

EXTRACT_ROOT = DATA_ROOT / "extracted"

vote_files = sorted(
    EXTRACT_ROOT.glob("*/raw_votes_parquet/*.parquet")
)

print("Votes file appearances:", len(vote_files))

Votes file appearances: 35658


## 2. Inspect the Votes data structure

### 2.1 Inspect one sample Votes file

In [2]:
votes_sample = pl.read_parquet(vote_files[0])

print("Shape:", votes_sample.shape)
print("Schema:", votes_sample.schema)

votes_sample.head(10)

Shape: (1, 3)
Schema: Schema({'match_id': Int64, 'home_vote': Int64, 'away_vote': Int64})


match_id,home_vote,away_vote
i64,i64,i64
11974053,272,61


## 3. Check essential data quality issues

### 3.1 Check schema consistency and missing values

In [3]:
schemas = [pl.read_parquet_schema(file) for file in vote_files]

print("Unique schemas:", len({tuple(schema.items()) for schema in schemas}))

votes_all = pl.concat([
    pl.read_parquet(file)
    for file in vote_files
])

votes_all.null_count()

Unique schemas: 1


match_id,home_vote,away_vote
u32,u32,u32
0,0,0


## 4. Build the full Votes dataset

### 4.1 Combine Votes files with snapshot date

In [4]:
votes_all = pl.concat([
    pl.read_parquet(file)
    .with_columns(
        pl.lit(file.parents[1].name)
        .str.strptime(pl.Date, "%Y%m%d")
        .alias("snapshot_date")
    )
    for file in vote_files
])

print("Shape:", votes_all.shape)
print(
    "Date range:",
    votes_all["snapshot_date"].min(),
    "to",
    votes_all["snapshot_date"].max()
)

Shape: (35658, 4)
Date range: 2024-02-01 to 2024-03-31


## 5. Validate the Votes dataset

### 5.1 Check for duplicate Vote records

In [5]:
duplicate_votes = (
    votes_all
    .group_by(["snapshot_date", "match_id"])
    .len()
    .filter(pl.col("len") > 1)
)

print("Duplicate Vote keys:", duplicate_votes.height)

Duplicate Vote keys: 0


### 5.2 Check for invalid vote counts

In [6]:
votes_all.select(
    (pl.col("home_vote") < 0).sum().alias("invalid_home_vote"),
    (pl.col("away_vote") < 0).sum().alias("invalid_away_vote"),
)

invalid_home_vote,invalid_away_vote
u32,u32
0,0


## 6. Save the processed data

### 6.1 Save the processed Votes data

In [7]:
PROCESSED_ROOT = DATA_ROOT / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

votes_output = PROCESSED_ROOT / "votes_clean.parquet"

votes_all.write_parquet(votes_output)

saved_votes = pl.read_parquet(votes_output)

print("Saved shape:", saved_votes.shape)
print("Exact match:", saved_votes.equals(votes_all))

Saved shape: (35658, 4)
Exact match: True


## Votes Preprocessing Summary

- Found **35,658** Votes file appearances.
- All files used the same schema.
- The dataset contains:
  - `match_id`
  - `home_vote`
  - `away_vote`
- No missing values were found.
- Combined all snapshots and added `snapshot_date`.
- Final dataset shape: **35,658 rows × 4 columns**.
- Snapshot dates range from **2024-02-01 to 2024-03-31**.
- No duplicate records were found for:
  `snapshot_date + match_id`.
- No negative `home_vote` or `away_vote` values were found.
- No rows needed to be removed or modified.
- Saved the cleaned dataset as `processed/votes_clean.parquet`.
- The saved file was read back successfully and matched the processed DataFrame exactly.